In [3]:
import numpy as np
from nnfs.datasets import spiral_data
import nnfs
nnfs.init()

# -----------------------------
# Dataset (1000 samples, 3 classes)
# Binary target: class 0 vs rest
# -----------------------------
X, y = spiral_data(samples=1000, classes=3)
y = (y == 0).astype(float)   # binary target

# -----------------------------
# Parameters (3 weights like w1, w2, w3)
# -----------------------------
w1, w2, w3 = 0.1, -0.2, 0.05
b = 0.0
lr = 0.01

# -----------------------------
# Functions (exact graph nodes)
# -----------------------------
def mul(x, w):
    return x * w

def summ(a, b, c, bias):
    return a + b + c + bias

def relu(x):
    return max(0.0, x)

def relu_derivative(x):
    return 1.0 if x > 0 else 0.0

def loss_fn(y_pred, y_true):
    return (y_pred - y_true) ** 2

# -----------------------------
# Training
# -----------------------------
for epoch in range(200):

    total_loss = 0.0

    for i in range(len(X)):
        x1, x2, x3 = X[i][0], X[i][1], X[i][0]  # reuse input for 3rd
        y_true = y[i]

        # ===== FORWARD PASS =====
        m1 = mul(x1, w1)
        m2 = mul(x2, w2)
        m3 = mul(x3, w3)

        s = summ(m1, m2, m3, b)
        r = relu(s)

        loss = loss_fn(r, y_true)
        total_loss += loss

        # ===== BACKWARD PASS (CHAIN RULE) =====
        # dLoss/dReLU
        dloss_dr = 2 * (r - y_true)

        # dReLU/dSum
        dr_ds = relu_derivative(s)

        # dLoss/dSum
        dloss_ds = dloss_dr * dr_ds

        # dSum/dMul = 1
        # dMul/dWi = Xi

        # dLoss/dWi
        dloss_dw1 = dloss_ds * x1
        dloss_dw2 = dloss_ds * x2
        dloss_dw3 = dloss_ds * x3

        # dLoss/db
        dloss_db = dloss_ds * 1.0

        # ===== UPDATE =====
        w1 -= lr * dloss_dw1
        w2 -= lr * dloss_dw2
        w3 -= lr * dloss_dw3
        b  -= lr * dloss_db

    if epoch % 20 == 0:
        print(f"Epoch {epoch}")
        print("Loss:", total_loss / len(X))
        print("w1, w2, w3:", w1, w2, w3)
        print("-" * 40)

Epoch 0
Loss: 0.017331683001351548
w1, w2, w3: 0.030716722332928922 -0.03934950526397364 -0.019283277667071077
----------------------------------------
Epoch 20
Loss: 0.3055019097002545
w1, w2, w3: 0.003928667068611848 -1.3430690479058427 -0.04607133293138858
----------------------------------------
Epoch 40
Loss: 0.3070180449493285
w1, w2, w3: 0.006346905226784447 -1.7176017983752565 -0.043653094773216
----------------------------------------
Epoch 60
Loss: 0.3077987758512262
w1, w2, w3: -0.0017849024939339245 -1.9113766145905005 -0.05178490249393431
----------------------------------------
Epoch 80
Loss: 0.3081644670892986
w1, w2, w3: 0.006483533527029301 -2.0492841993877264 -0.04351646647297099
----------------------------------------
Epoch 100
Loss: 0.3115233776060412
w1, w2, w3: -0.17784815373847013 -2.2140204443927542 -0.22784815373847117
----------------------------------------
Epoch 120
Loss: 0.31180563669130734
w1, w2, w3: -0.19470430205251024 -2.3697814582256 -0.2447043020525

## Below is the explanation of the code and the understanding of how the partial derivatives and the back propogation actually works 

Perfect 👍
Below is **exactly what you asked for**:
a **single `.md` (Markdown) file** that contains

* ✅ **code first**
* ✅ **deep explanation below each block**
* ✅ **chain-rule explanation exactly like your diagram**
* ✅ **why each step is needed**
* ✅ **easy to read, exam + learning friendly**

You can **copy–paste this entire answer into a file named**
👉 `manual_backprop_spiral.md`

---

# 🧠 Manual Backpropagation on NNFS Spiral Dataset

*(Code + Deep Explanation)*

---

## 📌 Problem Setup

We train **one neuron** with **3 weights** using:

* NNFS **spiral dataset**
* **1000 samples**
* **3 classes** (converted to binary)
* **ReLU activation**
* **Mean Squared Error loss**
* **Manual forward + backward propagation**

---

## 📊 Computation Graph (Conceptual Diagram)

![Image](https://colah.github.io/posts/2015-08-Backprop/img/tree-backprop.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1200/0%2ALnQvzEdc8wkkUte9.png)

![Image](https://miro.medium.com/1%2AW7ZPd1tvyi_cIdpoDdX3DA.png)

The data flows **forward** like this:

```
x → mul → sum → ReLU → loss
```

Gradients flow **backward** in the **reverse direction**.

---

## 🧾 Code Implementation

```python
import numpy as np
from nnfs.datasets import spiral_data
import nnfs
nnfs.init()

# -----------------------------
# Dataset
# -----------------------------
X, y = spiral_data(samples=1000, classes=3)

# Convert to binary target: class 0 vs rest
y = (y == 0).astype(float)

# -----------------------------
# Parameters (3 weights)
# -----------------------------
w1, w2, w3 = 0.1, -0.2, 0.05
b = 0.0
learning_rate = 0.01

# -----------------------------
# Node-level functions
# -----------------------------
def mul(x, w):
    return x * w

def summ(a, b, c, bias):
    return a + b + c + bias

def relu(x):
    return max(0.0, x)

def relu_derivative(x):
    return 1.0 if x > 0 else 0.0

def loss_fn(y_pred, y_true):
    return (y_pred - y_true) ** 2
```

---

## 🔍 Explanation of Forward Pass (WHY each step exists)

### 1️⃣ Multiplication Node (`mul`)

[
m_i = x_i \cdot w_i
]

**Why this step?**

* Weight controls **importance** of input
* Learning means **adjusting these weights**
* Without multiplication → no learning

---

### 2️⃣ Summation Node (`sum`)

[
s = m_1 + m_2 + m_3 + b
]

**Why sum?**

* Creates a **linear combination**
* Allows neuron to form **decision boundaries**

**Why bias?**

* Shifts the activation
* Without bias → neuron always passes through origin

---

### 3️⃣ ReLU Activation

[
r = \max(0, s)
]

**Why ReLU?**

* Adds **non-linearity**
* Blocks negative signals
* Prevents network from becoming purely linear

---

### 4️⃣ Loss Function

[
L = (r - y)^2
]

**Why loss?**

* Measures **how wrong** the prediction is
* Training goal = **minimize this value**

---

## 🔁 Training Loop (Forward + Backward)

```python
for epoch in range(200):

    total_loss = 0.0

    for i in range(len(X)):
        x1, x2, x3 = X[i][0], X[i][1], X[i][0]
        y_true = y[i]

        # ===== FORWARD PASS =====
        m1 = mul(x1, w1)
        m2 = mul(x2, w2)
        m3 = mul(x3, w3)

        s = summ(m1, m2, m3, b)
        r = relu(s)

        loss = loss_fn(r, y_true)
        total_loss += loss

        # ===== BACKWARD PASS =====
        dloss_dr = 2 * (r - y_true)
        dr_ds = relu_derivative(s)
        dloss_ds = dloss_dr * dr_ds

        dloss_dw1 = dloss_ds * x1
        dloss_dw2 = dloss_ds * x2
        dloss_dw3 = dloss_ds * x3
        dloss_db  = dloss_ds * 1.0

        # ===== UPDATE =====
        w1 -= learning_rate * dloss_dw1
        w2 -= learning_rate * dloss_dw2
        w3 -= learning_rate * dloss_dw3
        b  -= learning_rate * dloss_db

    if epoch % 20 == 0:
        print(f"Epoch {epoch}")
        print("Loss:", total_loss / len(X))
        print("Weights:", w1, w2, w3)
        print("-" * 40)
```

---

Perfect catch 👍
You’re right — for **Markdown math rendering**, the formulas **must be wrapped in `$ ... $` or `$$ ... $$`**.

Below is the **corrected section ONLY**, ready to **directly replace** your existing part in the `.md` file.

---

## 🧮 Backpropagation (EXACTLY like your diagram)

### 🔗 Chain Rule (core idea)

$$
\frac{\partial L}{\partial w_1}
===============================

\frac{\partial L}{\partial \text{ReLU}}
\cdot
\frac{\partial \text{ReLU}}{\partial \text{Sum}}
\cdot
\frac{\partial \text{Sum}}{\partial \text{Mul}}
\cdot
\frac{\partial \text{Mul}}{\partial w_1}
$$

This equation explains **why gradients are multiplied step-by-step** when flowing backward.

---

### 1️⃣ Loss → Output derivative

$$
L = (r - y)^2
$$

$$
\frac{\partial L}{\partial r} = 2(r - y)
$$

**Meaning:**

> “If the neuron output changes slightly, how much does the loss change?”

---

### 2️⃣ ReLU derivative (the gate)

$$
r = \text{ReLU}(s) = \max(0, s)
$$

$$
\frac{\partial r}{\partial s} =
\begin{cases}
1 & \text{if } s > 0 \
0 & \text{if } s \le 0
\end{cases}
$$

**Why this matters:**

* If the neuron is **inactive**, gradient becomes **zero**
* That neuron **does not learn** (dead ReLU)

---

### 3️⃣ Combine loss and ReLU (chain rule)

$$
\frac{\partial L}{\partial s}
=============================

\frac{\partial L}{\partial r}
\cdot
\frac{\partial r}{\partial s}
$$

This tells us **how much the summed input must change** to reduce loss.

---

### 4️⃣ Sum node derivative

$$
s = m_1 + m_2 + m_3 + b
$$

$$
\frac{\partial s}{\partial m_i} = 1
$$

**Reason:**
The sum node **passes gradients unchanged**.

---

### 5️⃣ Multiplication node derivative

$$
m_i = x_i \cdot w_i
$$

$$
\frac{\partial m_i}{\partial w_i} = x_i
$$

**Key intuition:**

* Larger input → larger weight update
* Zero input → no learning for that weight

---

## ✅ Correct way to write this in `.md`

### RULE (very important)

* Use **ONLY math inside `$$ ... $$`**
* Put **explanatory text outside**
* **Never use `=====` inside equations**

---

## ✅ FINAL, CORRECT `.md` VERSION (copy–paste)

Replace your entire **“Final gradient for each weight”** section with this 👇

---

## ✅ Final gradient for each weight

### For ( w_1 )

$$
\frac{\partial L}{\partial w_1}


\frac{\partial L}{\partial r}
\cdot
\frac{\partial r}{\partial s}
\cdot
\frac{\partial s}{\partial m_1}
\cdot
\frac{\partial m_1}{\partial w_1}
$$

Since,

$$
\frac{\partial s}{\partial m_1} = 1
\quad \text{and} \quad
\frac{\partial m_1}{\partial w_1} = x_1
$$

We get,

$$
\frac{\partial L}{\partial w_1}


\frac{\partial L}{\partial r}
\cdot
\frac{\partial r}{\partial s}
\cdot
x_1
$$

---

### Similarly, for ( w_2 )

$$
\frac{\partial L}{\partial w_2}


\frac{\partial L}{\partial r}
\cdot
\frac{\partial r}{\partial s}
\cdot
x_2
$$

---

### And for ( w_3 )

$$
\frac{\partial L}{\partial w_3}


\frac{\partial L}{\partial r}
\cdot
\frac{\partial r}{\partial s}
\cdot
x_3
$$

---

## 🔧 Weight update rule

$$
w_i = w_i - \eta \frac{\partial L}{\partial w_i}
$$

where ( \eta ) is the learning rate.

---

### 🔧 Weight update rule

$$
w_i = w_i - \eta \frac{\partial L}{\partial w_i}
$$

* Gradient points **towards higher loss**
* Subtracting moves **towards minimum**
* $\eta$ is the **learning rate**

---
